In [6]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing import sequence
import nltk
from nltk.stem import WordNetLemmatizer
import numpy as np

In [8]:
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ancij\AppData\Roaming\nltk_data...


Data Loading and Cleaning

In [10]:
data = pd.read_csv('twitter_data.csv')
data

,id,user,sentiment,tweet
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
...,...,...,...,...
74677,9200,Nvidia,Positive,Just realized that the Windows partition of my...
74678,9200,Nvidia,Positive,Just realized that my Mac window partition is ...
74679,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...
74680,9200,Nvidia,Positive,Just realized between the windows partition of...


In [11]:
data['sentiment'].value_counts()

sentiment
Negative      22542
Positive      20832
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64

In [12]:
data.isnull().sum()

id             0
user           0
sentiment      0
tweet        686
dtype: int64

In [13]:
data = data.dropna(axis=0)

In [14]:
def clean_text(text):
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = text.lower()
    text = ' '.join([lemmatizer.lemmatize(w) for w in text.split()])
    return text

In [15]:
data['tweet'] = data['tweet'].apply(clean_text)

C:\Users\ancij\AppData\Local\Temp\ipykernel_18400\2420922662.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['tweet'] = data['tweet'].apply(clean_text)


In [16]:
data

,id,user,sentiment,tweet
0,2401,Borderlands,Positive,im getting on borderland and i will murder you...
1,2401,Borderlands,Positive,i am coming to the border and i will kill you all
2,2401,Borderlands,Positive,im getting on borderland and i will kill you all
3,2401,Borderlands,Positive,im coming on borderland and i will murder you all
4,2401,Borderlands,Positive,im getting on borderland 2 and i will murder y...
...,...,...,...,...
74677,9200,Nvidia,Positive,just realized that the window partition of my ...
74678,9200,Nvidia,Positive,just realized that my mac window partition is ...
74679,9200,Nvidia,Positive,just realized the window partition of my mac i...
74680,9200,Nvidia,Positive,just realized between the window partition of ...


In [17]:
X = data['tweet']
Y = pd.get_dummies(data['sentiment'])

Train/Test Split

In [18]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42
)

Tokenization and Padding

In [19]:
max_tokens = 10000
maxlen = 64

tokenizer = Tokenizer(num_words=max_tokens)
tokenizer.fit_on_texts(X_train.values)

X_train = tokenizer.texts_to_sequences(X_train)
X_test = tokenizer.texts_to_sequences(X_test)

X_train = sequence.pad_sequences(X_train, maxlen=maxlen)
X_test = sequence.pad_sequences(X_test, maxlen=maxlen)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (59196, 64)
Test shape: (14800, 64)


In [23]:
from keras.models import Sequential
import keras
from keras.layers import (
    Dense,
    Embedding,
    Conv1D,
    MaxPooling1D,
    LSTM,
    Flatten,
    Bidirectional,
    GlobalMaxPooling1D,
    Dropout,
    BatchNormalization
)
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

Loading GloVe Embeddings

In [24]:
embeddings_index = {}

with open('glove.6B.100d.txt', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.array(values[1:], dtype='float32')
        embeddings_index[word] = coefs

print(f"Total words in GloVe: {len(embeddings_index)}")

Total words in GloVe: 400000


Embedding Matrix

In [25]:
embedding_dim = 100

embedding_matrix = np.zeros((max_tokens, embedding_dim))

for word, i in tokenizer.word_index.items():
    if i < max_tokens:
        vec = embeddings_index.get(word)
        if vec is not None:
            embedding_matrix[i] = vec

print("Embedding matrix shape:", embedding_matrix.shape)

Embedding matrix shape: (10000, 100)


Class Weights

In [26]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(np.argmax(Y_train.values, axis=1))
weights = compute_class_weight('balanced', classes=classes, y=np.argmax(Y_train.values, axis=1))
class_weight = dict(zip(classes, weights))

Building the Model

In [27]:
model = Sequential()

model.add(Embedding(
    input_dim=max_tokens,
    output_dim=embedding_dim,
    weights=[embedding_matrix],
    trainable=True
))

model.add(Conv1D(filters=64, kernel_size=3, padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.3))

model.add(Conv1D(filters=64, kernel_size=3, padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.3))

model.add(Bidirectional(LSTM(128, return_sequences=True, dropout=0.4, recurrent_dropout=0.3)))

model.add(GlobalMaxPooling1D())

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(4, activation='softmax'))

Model Training

In [28]:
model.compile(
    loss='categorical_crossentropy',
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
)

early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)

history = model.fit(
    X_train, Y_train,
    validation_split=0.2,
    batch_size=64,
    epochs=50,
    callbacks=[early_stopping, lr_scheduler],
    class_weight=class_weight
)

Epoch 1/50
740/740 ━━━━━━━━━━━━━━━━━━━━ 42s 49ms/step - accuracy: 0.3915 - loss: 1.3098 - precision: 0.6380 - recall: 0.0782 - val_accuracy: 0.5394 - val_loss: 1.1116 - val_precision: 0.7085 - val_recall: 0.2788 - learning_rate: 0.0010
Epoch 2/50
740/740 ━━━━━━━━━━━━━━━━━━━━ 31s 42ms/step - accuracy: 0.5447 - loss: 1.1379 - precision: 0.7158 - recall: 0.2768 - val_accuracy: 0.6031 - val_loss: 0.9853 - val_precision: 0.7304 - val_recall: 0.4633 - learning_rate: 0.0010
Epoch 3/50
740/740 ━━━━━━━━━━━━━━━━━━━━ 27s 36ms/step - accuracy: 0.6170 - loss: 0.9905 - precision: 0.7507 - recall: 0.4275 - val_accuracy: 0.6399 - val_loss: 0.8975 - val_precision: 0.7339 - val_recall: 0.5491 - learning_rate: 0.0010
Epoch 4/50
740/740 ━━━━━━━━━━━━━━━━━━━━ 27s 36ms/step - accuracy: 0.6859 - loss: 0.8463 - precision: 0.7913 - recall: 0.5545 - val_accuracy: 0.6752 - val_loss: 0.8443 - val_precision: 0.7468 - val_recall: 0.5997 - learning_rate: 0.0010
Epoch 5/50
740/740 ━━━━━━━━━━━━━━━━━━━━ 35s 47ms/step - 

Evaluation

In [29]:
preds = model.predict(X_test)

print(classification_report(
    np.argmax(Y_test.values, axis=1),
    np.argmax(preds, axis=1),
    target_names=Y_test.columns.tolist()
))

463/463 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step 
              precision    recall  f1-score   support

  Irrelevant       0.79      0.71      0.75      2696
    Negative       0.80      0.86      0.83      4380
     Neutral       0.77      0.74      0.75      3605
    Positive       0.79      0.81      0.80      4119

    accuracy                           0.79     14800
   macro avg       0.79      0.78      0.78     14800
weighted avg       0.79      0.79      0.79     14800



In [32]:
import pickle
model.save('sentiment_model.keras')
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)